# Delhi-NCR Quick Commerce Delivery Optimization

This notebook is the Jupyter version of the capstone project. Run each cell step by step.

## Tools Covered
- Python for data cleaning and analysis
- SQL-ready outputs
- Excel-ready outputs
- Power BI-ready dataset

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 2. Set File Paths

Make sure the raw CSV file exists at the path below.

In [ ]:
project_root = Path(r"C:\Users\caros\OneDrive\Documents\SQL project")
raw_path = Path(r"C:\Users\caros\Downloads\Delhi_NCR_Delivery_Operations_Raw.csv")

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

raw_path.exists()

## 3. Load Raw Dataset

In [ ]:
raw_df = pd.read_csv(raw_path)
df = raw_df.copy()
df.head()

## 4. Understand Dataset Size And Columns

In [ ]:
print("Raw rows and columns:", df.shape)
df.info()

## 5. Check Missing Values

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values

## 6. Remove Duplicates And Unwanted Columns

A clean dataset should not contain duplicate orders or privacy-sensitive/unwanted columns.

In [ ]:
print("Rows before duplicate removal:", len(df))
print("Duplicate Order_ID count:", df.duplicated("Order_ID").sum())

df = df.drop_duplicates(subset=["Order_ID"], keep="first").copy()

columns_to_drop = ["Legacy_Marketing_ID", "Temp_System_Log", "Customer_Phone_Raw"]
df = df.drop(columns=columns_to_drop)

print("Rows after duplicate removal:", len(df))
print("Columns after dropping unwanted fields:", len(df.columns))

## 7. Clean Rider Rating

`Rider_Rating_Raw` contains values like `Terrible`, `4.5 stars`, and `2.0`. We convert them into numbers.

In [ ]:
def parse_rating(value):
    if pd.isna(value):
        return np.nan
    
    text = str(value).strip().lower()
    rating_map = {
        "terrible": 1.0,
        "poor": 2.0,
        "average": 3.0,
        "good": 4.0,
        "excellent": 5.0,
    }
    
    if text in rating_map:
        return rating_map[text]
    
    match = re.search(r"\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else np.nan

df["Rider_Rating"] = df["Rider_Rating_Raw"].apply(parse_rating)
df[["Rider_Rating_Raw", "Rider_Rating"]].head(10)

## 8. Create Date And Time Columns

In [ ]:
df["Order_Timestamp"] = pd.to_datetime(df["Order_Timestamp"], errors="coerce")
df["Order_Date"] = df["Order_Timestamp"].dt.date
df["Order_Hour"] = df["Order_Timestamp"].dt.hour
df["Day_Name"] = df["Order_Timestamp"].dt.day_name()
df["Week"] = df["Order_Timestamp"].dt.isocalendar().week.astype("Int64")

df = df.dropna(subset=["Order_ID", "Order_Timestamp", "Dark_Store_ID", "SLA_Breached"])
df["Total_Items"] = df["Total_Items"].astype("int64")
df["Order_Value_INR"] = df["Order_Value_INR"].astype("float64")
df["SLA_Breached"] = df["SLA_Breached"].astype("int64")
df["Order_Hour"] = df["Order_Hour"].astype("int64")
df["Week"] = df["Week"].astype("int64")

df[["Order_Timestamp", "Order_Date", "Order_Hour", "Day_Name", "Week"]].head()

## 9. Create Business Features

In [ ]:
df["Delivery_Speed_KMPH"] = (df["Delivery_Distance_KM"] / (df["Transit_Time_Mins"] / 60)).replace([np.inf, -np.inf], np.nan)
df["Revenue_Per_KM"] = (df["Order_Value_INR"] / df["Delivery_Distance_KM"]).replace([np.inf, -np.inf], np.nan)

df["Delivery_Bucket"] = pd.cut(
    df["Total_Delivery_Time_Mins"],
    bins=[0, 10, 15, 20, 30, np.inf],
    labels=["0-10", "10-15", "15-20", "20-30", "30+"],
    right=False,
)

df["Distance_Bucket"] = pd.cut(
    df["Delivery_Distance_KM"],
    bins=[0, 1, 2, 3, 5, np.inf],
    labels=["0-1 km", "1-2 km", "2-3 km", "3-5 km", "5+ km"],
    right=False,
)

df[["Delivery_Speed_KMPH", "Revenue_Per_KM", "Delivery_Bucket", "Distance_Bucket"]].head()

## 10. Executive KPIs

In [ ]:
total_orders = len(df)
total_revenue = df["Order_Value_INR"].sum()
avg_order_value = df["Order_Value_INR"].mean()
avg_delivery_time = df["Total_Delivery_Time_Mins"].mean()
sla_breach_rate = df["SLA_Breached"].mean()
avg_rider_rating = df["Rider_Rating"].mean()

kpis = pd.DataFrame({
    "Metric": ["Total Orders", "Total Revenue", "Average Order Value", "Average Delivery Time", "SLA Breach Rate", "Average Rider Rating"],
    "Value": [total_orders, total_revenue, avg_order_value, avg_delivery_time, sla_breach_rate, avg_rider_rating]
})

kpis

## 11. Store Performance Analysis

In [ ]:
store_performance = (
    df.groupby("Dark_Store_ID")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue=("Order_Value_INR", "sum"),
        Avg_Order_Value=("Order_Value_INR", "mean"),
        Avg_Delivery_Time=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate=("SLA_Breached", "mean"),
        Avg_Distance=("Delivery_Distance_KM", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
    .sort_values("SLA_Breach_Rate", ascending=False)
)

store_performance

## 12. Local Rider Impact Analysis

This answers: Do local riders help reduce late deliveries?

In [ ]:
local_rider_impact = (
    df.groupby("Is_Local_Rider")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue=("Order_Value_INR", "sum"),
        Avg_Order_Value=("Order_Value_INR", "mean"),
        Avg_Delivery_Time=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate=("SLA_Breached", "mean"),
        Avg_Distance=("Delivery_Distance_KM", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
)

local_rider_impact

In [ ]:
local = local_rider_impact.set_index("Is_Local_Rider")

local_sla = local.loc["Yes", "SLA_Breach_Rate"]
non_local_sla = local.loc["No", "SLA_Breach_Rate"]
local_time = local.loc["Yes", "Avg_Delivery_Time"]
non_local_time = local.loc["No", "Avg_Delivery_Time"]

print(f"Local rider SLA breach rate: {local_sla:.2%}")
print(f"Non-local rider SLA breach rate: {non_local_sla:.2%}")
print(f"SLA improvement using local riders: {(non_local_sla - local_sla) * 100:.2f} percentage points")
print(f"Average time saving using local riders: {non_local_time - local_time:.2f} minutes per order")

## 13. Hourly Demand Analysis

In [ ]:
hourly_trend = (
    df.groupby("Order_Hour")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue=("Order_Value_INR", "sum"),
        Avg_Delivery_Time=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate=("SLA_Breached", "mean"),
    )
    .reset_index()
    .sort_values("Orders", ascending=False)
)

hourly_trend.head(10)

## 14. Delivery Root Cause Analysis

In [ ]:
delivery_drivers = (
    df.groupby(["Traffic_Density", "Weather", "Vehicle_Type"])
    .agg(
        Orders=("Order_ID", "count"),
        Avg_Delivery_Time=("Total_Delivery_Time_Mins", "mean"),
        SLA_Breach_Rate=("SLA_Breached", "mean"),
        Avg_Rider_Rating=("Rider_Rating", "mean"),
    )
    .reset_index()
    .sort_values(["SLA_Breach_Rate", "Orders"], ascending=[False, False])
)

delivery_drivers.head(10)

## 15. Category Revenue Analysis

In [ ]:
category_performance = (
    df.groupby("Primary_Category")
    .agg(
        Orders=("Order_ID", "count"),
        Revenue=("Order_Value_INR", "sum"),
        Avg_Order_Value=("Order_Value_INR", "mean"),
        SLA_Breach_Rate=("SLA_Breached", "mean"),
        Avg_Delivery_Time=("Total_Delivery_Time_Mins", "mean"),
    )
    .reset_index()
    .sort_values("Revenue", ascending=False)
)

category_performance

## 16. Export Clean Data And Summary Tables

In [ ]:
df.to_csv(processed_dir / "orders_clean_from_notebook.csv", index=False)
store_performance.to_csv(processed_dir / "store_performance_from_notebook.csv", index=False)
local_rider_impact.to_csv(processed_dir / "local_rider_impact_from_notebook.csv", index=False)
hourly_trend.to_csv(processed_dir / "hourly_trend_from_notebook.csv", index=False)
delivery_drivers.to_csv(processed_dir / "delivery_drivers_from_notebook.csv", index=False)
category_performance.to_csv(processed_dir / "category_performance_from_notebook.csv", index=False)

print("Files exported successfully to:", processed_dir)

## 17. Final Business Insights

- Local riders reduce SLA breach rate and save delivery time.
- Traffic and weather are major reasons for late deliveries.
- Store-level monitoring is needed, especially for high SLA breach stores.
- Peak-hour rider allocation can improve delivery performance.
- Power BI can be used to track these KPIs daily.